In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML



def plot_ekman_hodograph(wind_speed=10.0, wind_dir=0.0, latitude=30.0, Az=0.01, max_depth=200, elev=25, azim=-60):
    """
    绘制无限深海 Ekman 螺旋的 3D 矢端曲线 (Hodograph)，支持视角调节
    """
    # 1. 物理常数与边界保护
    Omega = 7.292e-5  # 地球自转角速度 (rad/s)
    rho_a = 1.2       # 空气密度 (kg/m^3)
    rho_w = 1025.0    # 海水密度 (kg/m^3)
    C_D = 1.5e-3      # 海面拖曳系数
    
    # 避免赤道 f=0 导致计算溢出
    if abs(latitude) < 5.0:
        latitude = 5.0 if latitude >= 0 else -5.0
    wind_speed = max(wind_speed, 0.5) # 避免风速为0
    
    phi = np.radians(latitude)
    f = 2 * Omega * np.sin(phi)
    sign_f = np.sign(f) # 北半球为 1，南半球为 -1
    
    # 2. 计算 Ekman 动力学参数
    tau = rho_a * C_D * wind_speed**2           # 风应力大小
    d = np.sqrt(2 * Az / abs(f))                # Ekman 衰减深度 (m)
    V0 = tau / (rho_w * np.sqrt(abs(f) * Az))   # 表面流速大小 (m/s)
    V0 = min(V0, 2.5)  # 限制最大流速，防止图像坐标崩坏
    
    # 3. 角度转换 (罗盘方位 -> 数学极坐标)
    theta_c = np.radians(wind_dir)
    theta_m = np.pi/2 - theta_c 
    
    # 表面流速方向 (北半球偏右45°，南半球偏左45°)
    theta_0 = theta_m - sign_f * np.pi/4
    
    # 4. 生成深度剖面数据
    z = np.linspace(0, max_depth, 500)
    decay = np.exp(-z / d)
    phase = theta_0 - sign_f * z / d
    
    u = V0 * decay * np.cos(phase)
    v = V0 * decay * np.sin(phase)
    
    # 5. 3D 绘图设置
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 坐标轴范围 (保持 X 和 Y 对称，确保螺旋线不变形)
    vel_limit = max(1.0, V0 * 1.2)
    ax.set_xlim([-vel_limit, vel_limit])
    ax.set_ylim([-vel_limit, vel_limit])
    ax.set_zlim([max_depth, 0]) # Z轴反转，0在顶部，符合海洋学深度习惯

    
    ax.set_xlabel('Eastward Velocity u (m/s)', labelpad=10, fontsize=11)
    ax.set_ylabel('Northward Velocity v (m/s)', labelpad=10, fontsize=11)
    ax.set_zlabel('Depth (m)', labelpad=10, fontsize=11)
    
    # 绘制连续的 Ekman 螺旋线 (按深度着色)
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(z)))
    for i in range(len(z)-1):
        ax.plot(u[i:i+2], v[i:i+2], z[i:i+2], color=colors[i], linewidth=2.5, alpha=0.8)
        
    # 绘制特定深度的流速矢量“辐条” (从 Z 轴指向螺旋线)
    step = max(1, len(z) // 15) # 均匀取 15 个深度层
    z_q, u_q, v_q = z[::step], u[::step], v[::step]
    
    for i in range(len(z_q)):
        ax.plot([0, u_q[i]], [0, v_q[i]], [z_q[i], z_q[i]], 'k--', alpha=0.4, linewidth=1)
        ax.scatter(u_q[i], v_q[i], z_q[i], c='k', s=30, depthshade=False)
        
    # 绘制海面 (Z=0) 参考圆
    theta_circle = np.linspace(0, 2*np.pi, 100)
    r_circle = vel_limit * 0.95
    ax.plot(r_circle*np.cos(theta_circle), r_circle*np.sin(theta_circle), 
            np.zeros_like(theta_circle), 'gray', linestyle=':', alpha=0.5)
            
    # 绘制海面风矢量 (红色箭头)
    wind_scale = (vel_limit * 0.8) / wind_speed 
    w_u = wind_speed * wind_scale * np.cos(theta_m)
    w_v = wind_speed * wind_scale * np.sin(theta_m)
    
    ax.plot([0, w_u], [0, w_v], [0, 0], 'r-', linewidth=3)
    ax.scatter(w_u, w_v, 0, c='red', s=80, marker='.', depthshade=False, label='Wind Direction')
    ax.text(w_u*1.1, w_v*1.1, 0, f'Wind\n{wind_speed}m/s', color='red', fontsize=10, ha='center')
    
    # 标题
    hemisphere = 'Northern Hemisphere (Right deflection)' if sign_f > 0 else 'Southern Hemisphere (Left deflection)'
    title = f'Ekman Spiral Hodograph (3D Velocity Profile)\n{hemisphere} | Lat: {latitude:.1f}° | Az: {Az:.1e} m²/s'
    ax.set_title(title, fontsize=13, pad=20, fontweight='bold')
    
    # 🌟 核心修改：使用传入的 elev 和 azim 参数设置视角
    ax.view_init(elev=elev, azim=azim)
    plt.tight_layout()
    plt.show()


    # --- 实时输出物理参数信息面板 ---
    hemisphere_str = "北半球 (向右偏转 45°)" if sign_f > 0 else "南半球 (向左偏转 45°)"
    info_html = f"""
    <div style="background-color: #f0f8ff; padding: 12px 20px; border-radius: 6px; border-left: 5px solid #1f77b4; margin-bottom: 15px; font-family: sans-serif; font-size: 14px;">
        <h4 style="margin-top: 0; margin-bottom: 10px; color: #1f77b4;">🌊 当前 Ekman 流场物理参数</h4>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px;">
            <div><b>科里奥利参数 (f):</b> {f:.2e} 1/s</div>
            <div><b>Ekman 深度尺度 (d):</b> {d:.2f} m</div>
            <div><b>表层流速大小 (V₀):</b> {V0:.4f} m/s</div>
            <div><b>海面风应力 (τ):</b> {tau:.4f} N/m²</div>
            <div style="grid-column: span 2;"><b>偏转效应:</b> {hemisphere_str}</div>
        </div>
    </div>
    """
    display(HTML(info_html))

# ==========================================
# 创建交互式控件
# ==========================================
w = widgets.interactive(
    plot_ekman_hodograph,
    wind_speed=widgets.FloatSlider(value=10.0, min=1.0, max=25.0, step=0.5, description='风速 (m/s)',style={'description_width': '100px'}),
    wind_dir=widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0, description='风向 ',style={'description_width': '100px'}),
    latitude=widgets.FloatSlider(value=30.0, min=-80.0, max=80.0, step=1.0, description='纬度 (°)',style={'description_width': '100px'}),
    Az=widgets.FloatLogSlider(value=0.01, base=10, min=-4.0, max=-1.0, step=0.1, description='湍粘性系数 Az (m²/s)',style={'description_width': '130px'}),
    max_depth=widgets.IntSlider(value=50, min=10, max=100, step=10, description='显示深度 (m)',style={'description_width': '100px'}),
    elev=widgets.IntSlider(value=25, min=0, max=90, step=1, description='俯仰角 (°)',style={'description_width': '100px'}),
    azim=widgets.IntSlider(value=-60, min=-180, max=180, step=1, description='方位角 (°)',style={'description_width': '100px'})
)

# 必须调用 display 才会渲染出调节框！
display(w)

interactive(children=(FloatSlider(value=10.0, description='风速 (m/s)', max=25.0, min=1.0, step=0.5, style=Slide…